In [0]:
data = [
  ("Manish",["Java","Scala","C++"]),
  ("rahul",["Spark","Java","C++","Spark","Java"]),
  ("shyam",["CSharp","VB","Spark","Python"])
]
columns=["name","language"]

df=spark.createDataFrame(data,columns)
df.show()
from pyspark.sql.functions import explode
df.select(df.name,explode(df.language)).show()

In [0]:
data =[(1,'abc@gmail.com'),(2,'def@gmail.com'),(1,'abc@gmail.com')]
column=['id','name']

df =spark.createDataFrame(data,column)
df.show()
df.createOrReplaceTempView("mytable")
#df_dist = spark.sql("select count(*),name from mytable group by name having count(*) >1")
#df_dist.show()
spark.sql("select distinct id,name from mytable group by id,name").show()
spark.sql("select id,name from mytable group by id,name").show()
spark.sql("with rn as (select id,name,row_number() over(partition by id order by id desc) as row from mytable ) select * from rn where row=1").show()

In [0]:
customer_data=[(1,'Manish'),(2,'Rahul'),(3,'Monu'),(4,'Ram')]
schema=["Customer_ID", "Customer_Name"]

order_data=[(1,4),(3,2)]
schema1=["Order_ID", "Customer_ID"]

df_customer=spark.createDataFrame(customer_data,schema)

df_order=spark.createDataFrame(order_data,schema1)
df_customer.show()
df_order.show()

df_customer.createOrReplaceTempView("customer_tb")
df_order.createOrReplaceTempView("order_tb")

spark.sql("select customer_tb.Customer_ID,customer_tb.Customer_Name from customer_tb join order_tb on customer_tb.Customer_ID==order_tb.Customer_ID").show()

spark.sql("select customer_tb.Customer_ID,customer_tb.Customer_Name from customer_tb left anti join order_tb on customer_tb.Customer_ID==order_tb.Customer_ID").show()

spark.sql("select * from customer_tb left join order_tb on customer_tb.Customer_ID=order_tb.Customer_ID where order_tb.Order_ID is null").show()

In [0]:

emp_data=[('Manish' , 1 , 75000),
('Raghav' , 1 , 85000 ),
('surya' , 1 , 80000 ),
('virat' , 2 , 70000),
('rohit' , 2 , 75000),
('jadeja' , 3 , 85000),
('anil' , 3 , 55000),
('sachin' , 3 , 55000),  
('zahir', 4, 60000),
('bumrah' , 4 , 65000) ]
schema= ["emp_name" ,"dept_id" ,"salary"]

dept_data = [(1, 'DATA ENGINEER'),(2, 'SALES'),(3, 'SOFTWARE'),(4, 'HR')]
schema1=['dept_id','dept_name']

emp_df=spark.createDataFrame(emp_data,schema)
dept_df=spark.createDataFrame(dept_data,schema1)

emp_df.createOrReplaceTempView("emp_tb")
dept_df.createOrReplaceTempView("dept_tb")
emp_df.show()
dept_df.show()

In [0]:



spark.sql("with cte as (select emp_name,dept_id,salary,row_number() over(partition by dept_id order by salary desc) as rn from emp_tb) select * from cte where rn=1 ").show()

In [0]:
spark.sql("with cte as (select emp_name,dept_id,salary,row_number() over(partition by dept_id order by salary desc) as rn from emp_tb) select emp_name from cte where rn=1 ").show()

In [0]:
spark.sql("select emp_name,dept_id,salary,row_number() over(partition by dept_id order by salary ) as rn from emp_tb ").show()

In [0]:
spark.sql("with cte as (select emp_name,dept_id,salary,row_number() over(partition by dept_id order by salary ) as rn from emp_tb) select emp_name,salary from cte where rn=1 ").show()

In [0]:
data1 = [(1, 'KKR'),(2, 'MI'),(3, 'RCB'),(4, 'GT')]
schema1=['team_id','team_name']
df_team=spark.createDataFrame(data1,schema1)
df_team.show()
df_team.createOrReplaceTempView("team_tb")

In [0]:
spark.sql("""
          with cte as (
select team_name,row_number() over(order by team_name) as rn  from team_tb
)select t1.team_name as team1,
		t2.team_name as team2
 from cte t1
 join cte t2
 on t1.rn<=t2.rn    
          """).show()

In [0]:
data = [(1,['mobile','PC','Tab']),(2,['mobile','PC']),(3,['Tab','Pen'])]
schema=['customer_id','product_purchase']

df=spark.createDataFrame(data,schema)
df.show()
from pyspark.sql.functions import *
df.select(df.customer_id,explode(df.product_purchase).alias("exp_pord")).show()


In [0]:
data=[(1, 'yes',None,None),(2, None,'yes',None),(3, 'No',None,'yes')]
schema=['customer_id','device_using1','device_using2','device_using3']

df=spark.createDataFrame(data,schema)
df.show()

In [0]:
from pyspark.sql.functions import *
df.withColumn("mycol",coalesce(col("device_using1"),col("device_using2"),col("device_using3"))).show()

In [0]:

data=[('Manish','{"street": "123 St", "city": "Delhi"}'),('Ram','{"street": "456 St", "city": "Mumbai"}')]
schema=['name','address']
df=spark.createDataFrame(data,schema)
print(df.printSchema())
display(df)

In [0]:
df.createOrReplaceTempView("data_tb")
spark.sql("select name,address,from_json(address,'street string,city string') as address_new from data_tb").show()

In [0]:
spark.sql("select * from data_tb").show()

In [0]:
spark.sql("""with cte as(
select name,address,from_json(address,'street string,city string') as address_new from data_tb
)   select *,address_new.street,address_new.city from cte
""").show()


In [0]:
data = [ ['2024-01-01',20000], ['2024-01-02',10000],[ '2024-01-03',150000], ['2024-01-04',100000], ['2024-01-05',210000]] 
  
#define column names
columns = ['date', 'sales'] 
  
#create dataframe using data and column names
df = spark.createDataFrame(data, columns) 
df.show()

In [0]:
df.createOrReplaceTempView("sales_tb")

In [0]:
spark.sql("select *,sum(sales) over(order by date) as sum,lag(sales) over(order by date) as prev_val,lead(sales) over(order by date) as next_val from sales_tb").show()
